# NCM para Português: jornada de treinamento piloto

Este notebook cria um caminho experimental para treinar Next-Concept Modeling em português brasileiro, sem substituir nem modificar o notebook `treino_ncm_codigo_python.ipynb`.

## Objetivo

Responder, com dados medidos:

1. o VQ-VAE oficial, treinado em inglês, pode representar português com boa reconstrução;
2. o gerador NCM aprende português geral, instrução e raciocínio em uma GPU RTX 3060 Ti;
3. a arquitetura mantém velocidade e coerência suficientes para justificar um piloto maior.

## Não objetivos

- não é a promessa de um assistente geral pela primeira execução;
- não é linguagem automática após instalar dependências;
- não substitui avaliação humana por perplexidade;
- não expõe o corpus ou publicação privada como artefato público;
- não executa nenhum treinamento até os flags serem ativados explicitamente.

## Gates antes do piloto completo

Siga a ordem. Não avance um gate sem dados registrados.

- **Gate A:** acentos e caracteres PT-BR são reconstruídos; smoke test sem erro.
- **Gate B:** o VQ-VAE adaptado melhora a reconstrução sobre o checkpoint XSum.
- **Gate C:** 300–500 milhões de tokens piloto reduzem perplexidade em Carolina.
- **Gate D:** o modelo passa em prompts curtos de instrução sem colapso ou repetição.
- **Gate E:** raciocínio curto é coerente em exemplos verificáveis.
- **Gate F:** VRAM fica abaixo de 8 GB e throughput é registrável.

Os três corpora são:

- `carolina-c4ai/carol-subcorpora`: português brasileiro geral;
- `cnmoro/smoltalk-555k-ptbr`: instrução/conversa em português;
- `cnmoro/reasoning-v1-20m-portuguese`: prompt, pensamento e resposta em português.

Comece com mistura piloto 70% Carolina, 20% SmolTalk e 10% raciocínio.

In [ ]:
from pathlib import Path
import ast
import gc
import importlib.util
import json
import random
import subprocess
import sys

import torch

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__)
print("device:", device)
if device.type != "cuda":
    raise RuntimeError("Este experimento exige CUDA; a execução foi bloqueada para evitar resultados lentos ou inconsistentes.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2), "GB")

if torch.cuda.get_device_properties(0).total_memory / 2**30 < 7.5:
    raise RuntimeError("Use uma GPU com pelo menos 8 GB de VRAM ou ajuste explicitamente o piloto.")

In [ ]:
required = ["omegaconf", "hydra", "datasets", "huggingface_hub", "transformers", "sentencepiece", "pyarrow"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    print("Dependências ausentes:", missing)
    raise RuntimeError("Instale as dependências antes de continuar.")
print("Dependências essenciais encontradas.")

## Limites seguros da 3060 Ti

O piloto começa pequeno mesmo que o hardware suporte mais. Os valores são um teto inicial:

- batch físico 2;
- acumulação 8;
- alvo máximo 128 a 256 tokens;
- BF16 quando suportado pela instalação/CUDA; FP16 como fallback;
- grad clipping 1.0;
- 300–500 milhões de tokens de pré-treino pilotados;
- early stopping por perda de validação;
- checkpoint por época e checkpoint `best.pt`.

Não carregue Carolina completa na memória. Use streaming ou shards locais.

In [ ]:
PILOT_TARGET_TOKENS = 300_000_000
PILOT_MAX_TARGET_TOKENS = 500_000_000
TRAIN_MAX_TOKENS_PER_EXAMPLE = 256
PREFIX_LENGTH = 512
MAX_OUTPUT_BLOCKS = 3
BATCH_SIZE = 2
ACCUMULATION_STEPS = 8
SEED = 42

MIXTURE = {
    "carolina": 0.70,
    "smoltalk": 0.20,
    "reasoning": 0.10,
}

OUTPUT_DIR = Path("/workspace/results/ncm_ptbr")
CHECKPOINT_DIR = Path("/workspace/models/ncm_ptbr")
DATA_DIR = Path("/workspace/data/ncm_ptbr")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("mistura:", json.dumps(MIXTURE, ensure_ascii=False))
print("meta piloto:", f"{PILOT_TARGET_TOKENS:,}" , "a", f"{PILOT_MAX_TARGET_TOKENS:,}" , "tokens")

## Checkout reproduzível e checkpoint oficial

O modelo oficial utilizado neste notebook é o checkpoint XSum. Ele fornece o gerador, o VQ-VAE e o vocabulário GPT-2 usados como inicialização. A adaptação ao português é feita por re-treinar o Stage 1.

In [ ]:
NCM_COMMIT = "c70274273f489abc73832b2b98584a8483b6b203"
NCM_DIR = Path("/workspace/models/ncm")

if not (NCM_DIR / ".git").exists():
    NCM_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "https://github.com/HazyResearch/ncm.git", str(NCM_DIR)], check=True)

subprocess.run(["git", "-C", str(NCM_DIR), "checkout", NCM_COMMIT], check=True)
sys.path.insert(0, str(NCM_DIR))
print("NCM commit:", NCM_COMMIT)

In [ ]:
from huggingface_hub import hf_hub_download

SOURCE_XSUM_DIR = Path("/workspace/models/ncm-checkpoints/xsum")
SOURCE_XSUM_DIR.mkdir(parents=True, exist_ok=True)

for filename in ("config.yaml", "generator.pt", "tokenizer.pt"):
    path = hf_hub_download(
        repo_id="hazyresearch/ncm-checkpoints",
        filename=f"xsum/{filename}",
        local_dir=SOURCE_XSUM_DIR.parent,
    )
    downloaded = Path(path)
    target = SOURCE_XSUM_DIR / filename
    if downloaded != target:
        downloaded.replace(target)

print("Checkpoint base:", SOURCE_XSUM_DIR)

## Preparação dos dados

Use Carolina via o subcorpus balanceado/deduplicado. Este notebook prefere Carol·(D+B) para reduzir duplicação, mas permite substituir por Carol·B.

Filtros mínimos:

- remover vazio ou menos de 20 caracteres;
- remover texto com repetição excessiva;
- remover metadados XML expostos indevidamente;
- remover registros com menos de 70% de caracteres portugueses esperados;
- normalizar Unicode sem destruir palavras ou pontuação;
- separar por documento, não por fragmentos de linha;
- medir e remover duplicatas exatas e quase duplicatas;
- não usar dados de validação no treinamento;
- registrar licença de cada fonte antes de publicar pesos.

In [ ]:
import re
import unicodedata

PT_CHARS = re.compile(r"[A-Za-zÀ-ÖØ-öø-ÿ0-9 .,;:!?%()\-\n/\"'@#$%&*_=+<>\[\]{}]")

def normalize_text(text):
    text = unicodedata.normalize("NFC", str(text or ""))
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def is_quality_text(text):
    text = normalize_text(text)
    if len(text) < 20:
        return False
    alphabetic = sum(ch.isalpha() for ch in text)
    if alphabetic == 0:
        return False
    expected = sum(bool(PT_CHARS.fullmatch(ch)) for ch in text)
    if expected / max(len(text), 1) < 0.70:
        return False
    words = re.findall(r"\b[^\W\d_]\S*\b", text.lower())
    if len(words) >= 20:
        unique_ratio = len(set(words)) / len(words)
        if unique_ratio < 0.20:
            return False
    if "<teiHeader" in text or "<?xml" in text:
        return False
    return True

print("Teste da filtragem:")
for sample in ["Esta é uma frase válida em português brasileiro.", " ", "<teiHeader>\n<texto>"]:
    print(is_quality_text(sample), repr(normalize_text(sample)))

In [ ]:
from datasets import load_dataset

PILOT_SAMPLE_LIMIT = {
    "carolina": 250_000,
    "smoltalk": 100_000,
    "reasoning": 100_000,
}

smoltalk = load_dataset("cnmoro/smoltalk-555k-ptbr", split="train", streaming=True)
reasoning = load_dataset("cnmoro/reasoning-v1-20m-portuguese", split="train", streaming=True)

print("SmolTalk: dataset streaming carregado")
print("Reasoning: 20.9M registros declarados pelo Hugging Face")
print("Limite piloto por fonte:", PILOT_SAMPLE_LIMIT)

### Conversão para exemplos NCM

A divisão dos dados segue três formatos:

- **plain:** texto Carolina;
- **instruction:** conversa completa com prefixo e resposta;
- **reasoning:** prompt, pensamento curto e resposta final.

O primeiro piloto não expõe o pensamento como resposta livre ilimitada. Os exemplos são truncados a 256 tokens de alvo.

In [ ]:
import hashlib

def format_instruction(messages):
    if isinstance(messages, str):
        return normalize_text(messages)
    output = []
    for message in messages:
        role = str(message.get("role", "user")).strip().lower()
        content = normalize_text(message.get("content", ""))
        if not content:
            continue
        role_label = {"system": "sistema", "user": "usuário", "assistant": "assistente"}.get(role, role)
        output.append(f"<{role_label}>\n{content}")
    return "\n\n".join(output)

def format_reasoning(prompt, thought, answer):
    prompt = normalize_text(prompt)
    thought = normalize_text(thought)
    answer = normalize_text(answer)
    if not prompt or not answer:
        return None
    if thought:
        thought = thought[:3000]
        return f"<usuário>\n{prompt}\n\n<assistente>\n<pensamento>\n{thought}\n</pensamento>\n{answer}"
    return f"<usuário>\n{prompt}\n\n<assistente>\n{answer}"

def deduplication_key(text):
    normalized = normalize_text(text).lower()
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()

print("Formatadores prontos.")

## Diagnóstico do tokenizador antes do Stage 1

O checkpoint XSum usa o GPT-2 como pre-tokenizador. Aqui medimos o teto de reconstrução antes de adaptar o VQ-VAE ao português.

Métricas:

- acurácia por bloco;
- reconstrução exata;
- taxa de texto decodível;
- acentos preservados;
- caracteres PT-BR preservados;
- taxa de mistura inglesa detectada em uma amostra manual.

In [ ]:
from omegaconf import OmegaConf

CONFIG_PATH = SOURCE_XSUM_DIR / "config.yaml"
GENERATOR_PATH = SOURCE_XSUM_DIR / "generator.pt"
TOKENIZER_PATH = SOURCE_XSUM_DIR / "tokenizer.pt"

CFG = OmegaConf.load(CONFIG_PATH)
CFG.tokenizer.vqvae.checkpoint_path = str(TOKENIZER_PATH)

from tokenization.vqvae_tokenizer import VQVAETokenizer
from models.next_concept import NextConceptTransformer

tk = VQVAETokenizer.from_config(CFG.tokenizer)
tk.vqvae.to(device).eval()
for parameter in tk.vqvae.parameters():
    parameter.requires_grad_(False)

net = NextConceptTransformer(vqvae_local=tk.vqvae, config=CFG.model).to(device)
checkpoint = torch.load(GENERATOR_PATH, map_location="cpu", weights_only=False)
net.load_state_dict(checkpoint["model"], strict=True)
net.eval()

print("Gerador:", round(sum(p.numel() for p in net.parameters()) / 1e6, 2), "M parâmetros")

In [ ]:
VQVAE_BLOCK_SIZE = int(tk.vqvae.context_length)
PAD_ID = int(CFG.model.prefix_pad_token_id)

def split_text_blocks(text, max_blocks=MAX_OUTPUT_BLOCKS):
    ids = tk.pre_tokenizer.encode(normalize_text(text))
    return [ids[start:start + VQVAE_BLOCK_SIZE]
            for start in range(0, min(len(ids), VQVAE_BLOCK_SIZE * max_blocks), VQVAE_BLOCK_SIZE)]

@torch.no_grad()
def evaluate_vqvae_ptbr(texts, max_blocks=MAX_OUTPUT_BLOCKS):
    tk.vqvae.eval()
    correct = total = exact = pt_exact = 0
    blocks = 0
    for text in texts:
        for block in split_text_blocks(text, max_blocks):
            padded = block + [tk.pre_tokenizer.pad_token_id] * (VQVAE_BLOCK_SIZE - len(block))
            source = torch.tensor(padded, dtype=torch.long, device=device).unsqueeze(0)
            reconstructed = tk.vqvae.decode(tk.vqvae.encode(source)).argmax(dim=-1)
            original = source[:, :len(block)]
            decoded = reconstructed[:, :len(block)]
            correct += int((decoded == original).sum())
            total += original.numel()
            exact += int(torch.equal(decoded, original))
            blocks += 1

            original_text = tk.pre_tokenizer.decode(original[0].tolist(), skip_special_tokens=True)
            reconstructed_text = tk.pre_tokenizer.decode(decoded[0].tolist(), skip_special_tokens=True)
            pt_exact += int(original_text == reconstructed_text)
    return {
        "token_accuracy": correct / max(total, 1),
        "exact_block_rate": exact / max(blocks, 1),
        "text_exact_rate": pt_exact / max(blocks, 1),
        "blocks": blocks,
    }

PTBR_DIAGNOSTIC = [
    "Ação: português brasileiro precisa preservar acentos como á, ç, ã e õ.",
    "Raciocínio: primeiro identifique as premissas e depois conclua a resposta.",
    "def saudacao(nome):\n    return f'Olá, {nome}! Está tudo bem?'"
]

VQVAE_PTBR_BASELINE = evaluate_vqvae_ptbr(PTBR_DIAGNOSTIC)
print("VQ-VAE XSum baseline em PT-BR:", VQVAE_PTBR_BASELINE)

## Stage 1: adaptação do VQ-VAE ao português

O Stage 1 treina encoder, decoder e quantizador com reconstrução português. Ele não é o treino do gerador.

O próximo piloto deve usar Carolina balanceada/deduplicada como corpus principal. O exemplo abaixo deixa o treinamento desligado por padrão.

In [ ]:
RUN_STAGE1_VQVAE_PTBR = False
VQVAE_TRAIN_EPOCHS = 8
VQVAE_BATCH_SIZE = 2
VQVAE_ACCUMULATION_STEPS = 8
VQVAE_LEARNING_RATE = 1e-4
VQVAE_MAX_TRAIN_ITEMS = 100_000
VQVAE_OUTPUT_DIR = OUTPUT_DIR / "stage1_vqvae"
VQVAE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

stage1_candidates = sorted(VQVAE_OUTPUT_DIR.glob("tokenizer_ptbr_epoch_*.pt"))
ADAPTED_VQVAE_CHECKPOINT = stage1_candidates[-1] if stage1_candidates else None
print("Stage 1 ativo:", RUN_STAGE1_VQVAE_PTBR)
print("Checkpoint adaptado:", ADAPTED_VQVAE_CHECKPOINT)

A célula abaixo implementa o Stage 1. Ela só altera pesos e grava checkpoints quando `RUN_STAGE1_VQVAE_PTBR=True`.

O corpus real deve ser fornecido pela etapa de streaming de Carolina. A estrutura usa blocos de 48 tokens do XSum; esta é uma abstração inicial que precisa ser substituída pelo comprimento efetivo do texto/carolina.

In [ ]:
from torch.nn import functional as F

if RUN_STAGE1_VQVAE_PTBR:
    if torch.cuda.is_bf16_supported():
        AUTOCAST_DTYPE = torch.bfloat16
    else:
        AUTOCAST_DTYPE = torch.float16
    tk.vqvae.train()
    for parameter in tk.vqvae.parameters():
        parameter.requires_grad_(True)

    optimizer = torch.optim.AdamW(tk.vqvae.parameters(), lr=VQVAE_LEARNING_RATE, weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda", enabled=AUTOCAST_DTYPE == torch.float16)
    history = []

    # Substitua esta lista por um carregador streaming de Carolina antes da execução real.
    stage1_texts = PTBR_DIAGNOSTIC
    stage1_blocks = []
    for text in stage1_texts:
        for block in split_text_blocks(text):
            padded = block + [tk.pre_tokenizer.pad_token_id] * (VQVAE_BLOCK_SIZE - len(block))
            stage1_blocks.append(padded)

    if not stage1_blocks:
        raise RuntimeError("Nenhum bloco Stage 1 foi preparado.")

    for epoch in range(1, VQVAE_TRAIN_EPOCHS + 1):
        random.Random(SEED + epoch).shuffle(stage1_blocks)
        optimizer.zero_grad(set_to_none=True)
        total_loss = 0.0
        steps = 0

        for start in range(0, len(stage1_blocks), VQVAE_BATCH_SIZE):
            ids = torch.tensor(stage1_blocks[start:start + VQVAE_BATCH_SIZE], dtype=torch.long, device=device)
            with torch.autocast(device_type="cuda", dtype=AUTOCAST_DTYPE):
                logits, vq_loss, _, utilization, _ = tk.vqvae(ids)
                reconstruction_loss = F.cross_entropy(
                    logits.reshape(-1, logits.shape[-1]),
                    ids.reshape(-1),
                    ignore_index=tk.pre_tokenizer.pad_token_id,
                )
                loss = (reconstruction_loss + vq_loss) / VQVAE_ACCUMULATION_STEPS

            scaler.scale(loss).backward()
            total_loss += loss.item() * VQVAE_ACCUMULATION_STEPS
            steps += 1

            if steps % VQVAE_ACCUMULATION_STEPS == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(tk.vqvae.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

        if steps % VQVAE_ACCUMULATION_STEPS != 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        metrics = evaluate_vqvae_ptbr(PTBR_DIAGNOSTIC)
        history.append({"epoch": epoch, "loss": total_loss / max(steps, 1), **metrics})
        torch.save({
            "model": tk.vqvae.state_dict(),
            "epoch": epoch,
            "history": history,
            "baseline": VQVAE_PTBR_BASELINE,
        }, VQVAE_OUTPUT_DIR / f"tokenizer_ptbr_epoch_{epoch}.pt")
        print("Stage 1:", history[-1])

    ADAPTED_VQVAE_CHECKPOINT = VQVAE_OUTPUT_DIR / f"tokenizer_ptbr_epoch_{VQVAE_TRAIN_EPOCHS}.pt"
else:
    print("Stage 1 desativado. Nenhum peso foi atualizado.")

## Carregamento adaptado para o Stage 2

Sem checkpoint Stage 1 adaptado, o notebook usa o VQ-VAE XSum. Isso é útil somente para diagnosticar o gap e não deve ser chamado de modelo português treinado.

In [ ]:
if ADAPTED_VQVAE_CHECKPOINT is not None and ADAPTED_VQVAE_CHECKPOINT.exists():
    saved = torch.load(ADAPTED_VQVAE_CHECKPOINT, map_location="cpu", weights_only=False)
    tk.vqvae.load_state_dict(saved["model"], strict=True)
    print("VQ-VAE PT-BR carregado:", ADAPTED_VQVAE_CHECKPOINT)
else:
    print("Usando VQ-VAE XSum como baseline. Não avance ao Stage 2 sem Gate B.")

tk.vqvae.eval()
for parameter in tk.vqvae.parameters():
    parameter.requires_grad_(False)

net = NextConceptTransformer(vqvae_local=tk.vqvae, config=CFG.model).to(device)
base = torch.load(GENERATOR_PATH, map_location="cpu", weights_only=False)
net.load_state_dict(base["model"], strict=True)
net.eval()
print("Gerador base recarregado.")

## Stage 2: geração condicionada em português

Neste estágio, o gerador aprende a hierarquia condicional a partir de:

- prefixos e textos Carolina;
- perguntas e respostas SmolTalk;
- prompts, pensamentos e respostas verificáveis de raciocínio.

A ativação é explícita. Por padrão o treino não roda.

In [ ]:
RUN_STAGE2_GENERATOR_PTBR = False
GENERATOR_EPOCHS = 3
GENERATOR_PATIENCE = 2
GENERATOR_LEARNING_RATE = 5e-5
GENERATOR_MAX_STEPS = 5_000
GENERATOR_OUTPUT_DIR = OUTPUT_DIR / "stage2_generator"
GENERATOR_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Stage 2 ativo:", RUN_STAGE2_GENERATOR_PTBR)
print("Máximo inicial de steps:", GENERATOR_MAX_STEPS)

In [ ]:
def split_long_text(text, max_tokens=TRAIN_MAX_TOKENS_PER_EXAMPLE):
    ids = tk.pre_tokenizer.encode(normalize_text(text))
    if not ids:
        return []
    return [ids[start:start + max_tokens] for start in range(0, len(ids), max_tokens)]

def padded_block(ids):
    ids = ids[:VQVAE_BLOCK_SIZE]
    return ids + [PAD_ID] * (VQVAE_BLOCK_SIZE - len(ids))

@torch.no_grad()
def smoke_training_batch(prompt="Explique em português o que é raciocínio.", target="Raciocínio é o processo de organizar premissas para chegar a uma conclusão."):
    prompt_ids = tk.pre_tokenizer.encode(prompt)[:net.prefix_len]
    prompt_ids += [PAD_ID] * (net.prefix_len - len(prompt_ids))
    prefix_tensor = torch.tensor(prompt_ids, dtype=torch.long, device=device).unsqueeze(0)

    target_ids = padded_block(tk.pre_tokenizer.encode(target)[:VQVAE_BLOCK_SIZE])
    target_tensor = torch.tensor(target_ids, dtype=torch.long, device=device).unsqueeze(0)
    target_codes = tk.vqvae.encode(target_tensor)
    ncm_input = tk.vqvae.vq.idx_to_ncm_input(target_codes)

    prefix_embedding = tk.vqvae.encoder.wte(prefix_tensor)
    model_input = torch.cat([prefix_embedding, ncm_input], dim=1)
    logits = net.forward(model_input)
    return model_input, target_codes, logits

smoke_input, smoke_target, smoke_logits = smoke_training_batch()
print("entrada:", tuple(smoke_input.shape))
print("alvos:", tuple(smoke_target.shape))
print("logits:", tuple(smoke_logits.shape))
assert smoke_logits.shape[:2] == smoke_target.shape
print("Smoke test Stage 2: OK")

In [ ]:
if RUN_STAGE2_GENERATOR_PTBR:
    if ADAPTED_VQVAE_CHECKPOINT is None:
        print("AVISO: o Gate B ainda não foi aprovado. O treino usará o VQ-VAE XSum.")

    if torch.cuda.is_bf16_supported():
        autocast_dtype = torch.bfloat16
    else:
        autocast_dtype = torch.float16

    optimizer = torch.optim.AdamW(net.parameters(), lr=GENERATOR_LEARNING_RATE, weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda", enabled=autocast_dtype == torch.float16)
    history = []
    best_loss = float("inf")
    bad_epochs = 0

    for epoch in range(1, GENERATOR_EPOCHS + 1):
        net.train()
        optimizer.zero_grad(set_to_none=True)
        total_loss = 0.0
        steps = 0

        # Esta célula é um contorno seguro: execute o loop real com shards preparados e
        # streaming antes de ativar PILOT_MAX_TARGET_TOKENS.
        print("Treino real pendente: prepare os shards Carolina/SmolTalk/raciocínio.")
        break
else:
    print("Stage 2 desativado. Nenhum peso do gerador foi atualizado.")

## Plano de shards e mistura

A implementação completa deve criar três iteradores independentes e amostrar com os pesos da mistura 70/20/10. O uso de streaming evita ocupar RAM.

Cada shard deve conter:

```json
{"source":"carolina|smoltalk|reasoning","prefix":"...","target":"...","quality":"...","dedup_key":"..."}
```

O split recomendado:

- treino: 94%;
- validação: 3%;
- teste: 3%;
- sem overlap por documento ou quase duplicata;
- 300 milhões de tokens na primeira rodada;
- até 500 milhões somente após Gate C.

Em Carolina, prefira um subcorpus balanceado/deduplicado; em raciocínio, amostre e filtre antes do tokenizer.

In [ ]:
def write_dataset_manifest(path=DATA_DIR / "manifest.json"):
    manifest = {
        "seed": SEED,
        "mixture": MIXTURE,
        "pilot_target_tokens": PILOT_TARGET_TOKENS,
        "pilot_max_tokens": PILOT_MAX_TARGET_TOKENS,
        "max_target_tokens_per_example": TRAIN_MAX_TOKENS_PER_EXAMPLE,
        "prefix_length": PREFIX_LENGTH,
        "sources": {
            "carolina": "carolina-c4ai/carol-subcorpora",
            "smoltalk": "cnmoro/smoltalk-555k-ptbr",
            "reasoning": "cnmoro/reasoning-v1-20m-portuguese",
        },
        "stage1": RUN_STAGE1_VQVAE_PTBR,
        "stage2": RUN_STAGE2_GENERATOR_PTBR,
        "hardware": torch.cuda.get_device_name(0),
        "hardware_vram_gb": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2),
    }
    path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Manifesto:", path)

write_dataset_manifest()

## Avaliação obrigatória

Avalie em quatro grupos separados:

1. **Carolina:** perplexidade/loss em documentos não vistos.
2. **SmolTalk:** aderência à instrução e qualidade humana em 1.000 prompts não vistos.
3. **Raciocínio:** respostas corretas em problemas verificáveis; não pontue apenas o texto do pensamento.
4. **Portuguesidade:** acentos, gramática, repetição e mistura de inglês.

Mantenha conjuntos congelados. Não use dados de validação para ajustar prompts ou seleção de corpus.

In [ ]:
def repetition_score(text):
    text = normalize_text(text)
    words = re.findall(r"\b\w+\b", text.lower())
    if len(words) < 10:
        return 0.0
    grams = [tuple(words[i:i+3]) for i in range(len(words)-2)]
    if not grams:
        return 0.0
    return 1.0 - len(set(grams)) / len(grams)

COMMON_ENGLISH = re.compile(r"\b(the|and|with|without|this|that|because|however|answer|question)\b", re.I)

def evaluate_text_quality(text):
    text = normalize_text(text)
    return {
        "chars": len(text),
        "repetition_3gram": round(repetition_score(text), 4),
        "english_marker_hits": len(COMMON_ENGLISH.findall(text)),
        "has_accent": any(ch in text for ch in "áàâãéêíóôõúüç"),
    }

print(evaluate_text_quality("Este é um exemplo de texto em português brasileiro com acentuação."))

## Inferência interativa segura

A célula abaixo usa o gerador e o VQ-VAE atuais. Ela não salva pesos nem altera o notebook NCM Python.

In [ ]:
INFERENCE_PROMPT = "Explique a diferença entre uma hipótese e uma conclusão em termos simples."
INFERENCE_TEMPERATURE = 0.7
INFERENCE_TOP_K = 40
INFERENCE_TOP_P = 0.9
INFERENCE_BLOCKS = 2

@torch.no_grad()
def generate_ptbr(prompt, blocks=INFERENCE_BLOCKS, temperature=INFERENCE_TEMPERATURE):
    generated_parts = []
    net.eval()
    for block_index in range(blocks):
        condition = prompt
        if generated_parts:
            condition += "\n\nContinue a resposta em português:\n" + "".join(generated_parts)

        ids = tk.pre_tokenizer.encode(condition)[:net.prefix_len]
        ids += [PAD_ID] * (net.prefix_len - len(ids))
        prefix = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
        concept_ids = net.generate(
            idx=None,
            prefix_tokens=prefix,
            temperature=temperature,
            top_k=INFERENCE_TOP_K,
            top_p=INFERENCE_TOP_P,
        )
        logits = tk.vqvae.decode(concept_ids)
        raw_ids = logits.squeeze(0).argmax(dim=-1)
        generated_parts.append(tk.pre_tokenizer.decode(raw_ids.tolist(), skip_special_tokens=True))
    return "".join(generated_parts)

print("Prompt:", INFERENCE_PROMPT)
generated = generate_ptbr(INFERENCE_PROMPT)
print("Resposta:\n", generated)
print("Qualidade:", evaluate_text_quality(generated))

## Benchmark de memória e throughput

Faça três medições:

- VRAM pico com batch 1;
- VRAM pico com batch 2;
- tempo por exemplo e tokens por segundo.

Não declare aceleração sem medir na RTX 3060 Ti. Use pelo menos 30 amostras aquecidas e 100 amostras cronometradas.

In [ ]:
import time

@torch.no_grad()
def benchmark_smoke(samples=30, blocks=2):
    torch.cuda.reset_peak_memory_stats()
    prompt = "Explique uma ideia em português de forma clara."
    start = time.perf_counter()
    for _ in range(samples):
        _ = generate_ptbr(prompt, blocks=blocks, temperature=0.7)
    elapsed = time.perf_counter() - start
    peak_gb = torch.cuda.max_memory_allocated() / 2**30
    return {
        "samples": samples,
        "elapsed_s": round(elapsed, 4),
        "seconds_per_sample": round(elapsed / samples, 4),
        "peak_vram_gb": round(peak_gb, 3),
    }

print("Benchmark desligado por padrão. Execute benchmark_smoke(30, 2) manualmente.")

## Critérios de avanço

### Aprovar piloto maior

- VQ-VAE PT-BR melhora acurácia sobre o baseline XSum;
- VRAM permanece abaixo de 7,5 GB em batch 2;
- perplexidade Carolina cai de forma estável;
- menos de 10% dos exemplos apresentam repetição dominante em uma amostra de 100;
- menos de 5% apresentam mistura inglesa indevida em prompts em português;
- pelo menos 80% das respostas de instrução passam revisão humana básica;
- pelo menos 50% dos problemas verificáveis curtos ficam corretos antes de autorizar raciocínio longo;
- nenhum dado do notebook NCM Python foi substituído.

### Não avançar

- quando os pesos do Stage 1 forem instáveis;
- quando reconstrução da Carolina piorar;
- quando OOM aparecer no Stage 2;
- quando a mistura provocar colapso em português;
- quando o pensamento não melhorar resultados de resposta final.

Este primeiro notebook é um plano operacional. O loop completo Stage 1/Stage 2 deve ser preenchido com shards reais, dividido por fonte e medido antes de ativar o limite de 300–500 milhões de tokens.